# Fase 4 — Fine-tuning no Google Colab

Este notebook executa o **fine-tuning do SLM** (Llama 3.2 3B ou similar) com **Unsloth** (SFT + LoRA) no dataset da Fase 3 e exporta o modelo para **.gguf**, para uso local (Ollama/llama.cpp) sem custo de API.

**Por que Colab?** O treino exige GPU com CUDA e ~8 GB+ VRAM. Sem GPU local, usamos o ambiente do Colab (Design Decision 20). O mesmo código roda no repositório com `scripts/run_finetuning.py` se você tiver GPU.

**O que cada parte faz:**
1. **Verificar GPU** — Colab oferece T4 (16 GB); ative "Runtime → Change runtime type → GPU".
2. **Instalar Unsloth** — Biblioteca otimizada para treino 2x mais rápido e menos memória.
3. **Preparar dataset** — Subir os JSONL da Fase 3 ou clonar o repo.
4. **Treinar** — Carregar modelo base, aplicar LoRA, treinar e salvar checkpoints.
5. **Exportar .gguf** — Mesclar LoRA e salvar em GGUF.
6. **Salvar no Drive ou download** — Para usar o .gguf na Fase 5 no seu PC.

## 1. Verificar GPU

Sem GPU, o treino será inviável (muito lento). No Colab: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import torch
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## 2. Instalar Unsloth (otimizado para Colab)

O pacote `unsloth[colab-new]` inclui o wheel compatível com o ambiente Colab. A instalação pode levar alguns minutos.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install --no-deps xformers trl peft accelerate bitsandbytes --quiet

## 3. Preparar dataset

**Opção A — Clonar o repositório:** Se o dataset da Fase 3 já estiver no repo (em `data/datasets/`), clone e use esses arquivos.

**Opção B — Upload manual:** Faça upload de `nasa_se_synthetic_train.jsonl` e `nasa_se_synthetic_val.jsonl` para `/content/datasets/` (crie a pasta e suba os arquivos).

Execute **apenas uma** das opções abaixo.

In [ ]:
# --- Opção B: usar arquivos enviados para /content/datasets/ ---
# Se você fez upload para /content/datasets/, descomente e use:
# import os
# WORKDIR = "/content"
# TRAIN_PATH = "/content/datasets/nasa_se_synthetic_train.jsonl"
# VAL_PATH   = "/content/datasets/nasa_se_synthetic_val.jsonl"
# OUTPUT_DIR = "/content/output_fase4"
# os.makedirs(OUTPUT_DIR, exist_ok=True)
# import sys
# sys.path.insert(0, WORKDIR)  # se o código estiver em /content/rag-nasa, use sys.path.insert(0, "/content/rag-nasa")
pass

In [ ]:
# --- Opção A: clonar o repositório (substitua REPO_URL pela URL do seu repo) ---
import os
REPO_URL = "https://github.com/diegoluchetti/rag-nasa.git"
if not os.path.exists("/content/rag-nasa"):
    !git clone --depth 1 {REPO_URL} /content/rag-nasa
WORKDIR = "/content/rag-nasa"
TRAIN_PATH = f"{WORKDIR}/data/datasets/nasa_se_synthetic_train.jsonl"
VAL_PATH   = f"{WORKDIR}/data/datasets/nasa_se_synthetic_val.jsonl"
OUTPUT_DIR = "/content/output_fase4"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Train:", TRAIN_PATH, "existe:", os.path.exists(TRAIN_PATH))
print("Val:", VAL_PATH, "existe:", os.path.exists(VAL_PATH))

## 4. Treinar e exportar .gguf

Adicionamos o repositório ao `sys.path` para importar `src.finetuning`. Em seguida:
- Carregamos o dataset (formato Alpaca: instruction + output).
- Treinamos com Unsloth (LoRA, 3 épocas, QLoRA para caber na T4).
- Exportamos o modelo para GGUF (q4_k_m para menor tamanho).

In [ ]:
import sys
if "WORKDIR" in dir() and os.path.exists(WORKDIR):
    sys.path.insert(0, WORKDIR)
else:
    sys.path.insert(0, "/content/rag-nasa")

from pathlib import Path
from src.finetuning.data_loader import load_dataset_for_unsloth
from src.finetuning.train import run_training
from src.finetuning.export_gguf import merge_and_export_gguf

train_ds, val_ds = load_dataset_for_unsloth(TRAIN_PATH, VAL_PATH)
print("Train:", len(train_ds), "| Val:", len(val_ds))

trainer = run_training(
    train_ds, val_ds, OUTPUT_DIR,
    model_name="unsloth/Llama-3.2-3B-Instruct",
    use_qlora=True,
    max_seq_length=1024,
    lora_rank=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    learning_rate=2e-4, epochs=3,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    logging_steps=10, save_steps=100, save_strategy="epoch",
)

gguf_path = merge_and_export_gguf(trainer, trainer.tokenizer, Path(OUTPUT_DIR) / "gguf", quantization_method="q4_k_m")
print("GGUF salvo em:", gguf_path)

## 5. Salvar no Google Drive ou fazer download

Para não perder o modelo ao fechar o Colab, copie o .gguf para o Drive ou baixe para o seu PC.

In [ ]:
# --- Ou fazer download do .gguf para o seu PC ---
from google.colab import files
p = Path(gguf_path)
if p.is_file():
    files.download(str(p))
else:
    for f in p.glob("*.gguf"):
        files.download(str(f))
        break